<a href="https://colab.research.google.com/github/nya-mura/WAY-AI/blob/main/embedding_neural_net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import json

In [2]:
training_data = [
      ("this is amazing", 1),                                    ("i love this product", 1),                                ("absolutely wonderful experience", 1),                    ("great quality and fast delivery", 1),                    ("i am very happy with this", 1),
          ("excellent work keep it up", 1),                          ("best purchase i ever made", 1),                          ("fantastic service highly recommend", 1),                 ("really good and very useful", 1),                        ("superb quality loved it", 1),                            ("this is terrible", 0),
              ("i hate this so much", 0),                                ("worst experience ever", 0),                              ("absolutely awful and disgusting", 0),                    ("very bad quality do not buy", 0),                        ("horrible service never again", 0),
                  ("complete waste of money", 0),
                      ("disappointing and broken on arrival", 0),
                          ("really poor and useless product", 0),
                              ("garbage total garbage", 0),
                              ]


In [3]:
def build_vocab(data):
  vocab = {}
  idx = 0
  for sentence, _ in data:
    for word in sentence.lower().split():
      if word not in vocab:
        vocab[word] = idx
        idx += 1
  return vocab



SyntaxError: invalid syntax (1770285657.py, line 5)

In [ ]:
vocab = build_vocab(training_data)
vocab_size = len(vocab)

In [ ]:
def sentence_to_indices(sentence, vocab):
  indices = []
  word = sentence.lower().split()
  for w in word:
    if w in vocab:
      indices.append(vocab[w])
  return indices


In [ ]:
sentence_indices = []
for l, _ in training_data:
  sentence_indices.append(sentence_to_indices(l, vocab))
print(f"{sentence_indices}")


In [ ]:
print(f"{vocab}")

In [ ]:
labels = []
for _, label in training_data:
  labels.append(label)
print(f"{labels}")
labels = torch.tensor(labels, dtype=torch.float32)
print(f"{labels}")

In [ ]:
EMBED_DIM  = 4
HIDDEN_SIZE = 8

In [ ]:
class Embedding(nn.Module):
  def __init__(self, vocab_size, embed_dim, hidden_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size,embed_dim)
    self.layer1 = nn.Linear(embed_dim, hidden_size)
    self.relu = nn.ReLU()
    self.layer2 = nn.Linear(hidden_size, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, indices):
    embed = self.embedding(indices)
    x = embed.mean(dim=0)
    out = self.layer1(x)
    out = self.relu(out)
    out = self.layer2(out)
    out = self.sigmoid(out)
    return out

In [ ]:
model = Embedding(vocab_size, EMBED_DIM, HIDDEN_SIZE)
criterion = nn.BCELoss()
optimiser = torch.optim.SGD(model.parameters(), lr=0.05)


In [ ]:
epochs = 500
for epoch in range(epochs):


  for i, indices in enumerate(sentence_indices):
    idx_tensor = torch.tensor(indices, dtype=torch.long)
    label = labels[i].unsqueeze(0)
    predicted = model(idx_tensor).unsqeeze(0)
    l = criterion(predicted, label)
    if epoch % 100 == 0:
      print(f"{epoch} {l.items()}")
    optimiser.zero_grad()
    l.backward()
    optimiser.step()


In [ ]:
torch.save(model.state_dict(), "model_emb.pth")
with open("vocab.json", "w") as f:
  json.dump(vocab, f)



In [ ]:
model.eval()
def predict(sentence):
  indices = sentence_to_indices(sentence, vocab)
  idx_tensor = torch.tensor(indices, dtype=torch.long)
  with torch.no_grad():
    prob = model(idx_tensor).items()
  label = "idk"
  if prob >= 0.5:
    label = "good"
  else:
    label = "bad"
  return label, prob

In [ ]:
while True:
  sentence = input("enter")
  out = predict(sentence)
  print(f"> {out}")